## Code to preprocess Chenchen Zhu's datafrom the large intestine using 3D model, 2D layers, and RNA molecules

## Install and import libraries

In [77]:
%pip install requests pandas ipywidgets hra_jupyter_widgets

import os
import requests
import pandas as pd


Note: you may need to restart the kernel to use updated packages.


In [78]:
import ipywidgets as widgets

# Import hra-jupyter-widgets. For documentation, please see https://github.com/x-atlas-consortia/hra-jupyter-widgets/blob/main/usage.ipynb
from hra_jupyter_widgets import (
    BodyUi,               # CCF 3D Body UI
    CdeVisualization,     # Cell Distance Explorer (CDE)
    Eui,                  # Exploration User Interface (EUI)
    EuiOrganInformation,  # Organ Information
    FtuExplorer,          # Functional Tissue Unit (FTU) Explorer
    FtuExplorerSmall,     # FTU Explorer (small version)
    MedicalIllustration,  # Medical Illustration Viewer (2D FTU Viewer)
    ModelViewer,          # GLB model viewer
    NodeDistVis,          # VCCF 3D Node Distance Visualization
    Rui,                  # Registration User Interface (RUI)
)

## Download and save layers locally, then as DataFrames

In [79]:

# Make sure the data folder is present
folder = "data"

if not os.path.exists(folder):
    os.makedirs(folder)
    print(f"Folder '{folder}' created.")
else:
    print(f"Folder '{folder}' already exists.")

Folder 'data' already exists.


In [80]:
# load data from HRA CDN
def download_layer(layer_number:int):
  """Downloads the corresponding layer nodes CSV from the HRA CDN

  Args:
      layer_number (int): Number of layer
      
  Returns:
      DataFrame
  """

  url = f'https://cdn.humanatlas.io/image-store/vccf-data-cell-nodes/unpublished/colon-xenium-stanford/layer_{layer_number}-nodes.csv'
  
  dtype = {'x': 'float', 'y': 'float', 'Cell Type': 'string'}
  
  response = requests.get(url)
  
  file = f'layer_{layer_number}.csv'
  file_path = f'{folder}/{file}'
  
  if response.status_code == 200:
    print(f'Found {url}, downloading now.')
    
        # Check if the file exists
    if not os.path.exists(file_path):
        # If the file doesn't exist, run the curl command
        df = pd.read_csv(url, dtype=dtype)
        df.to_csv(f'{file_path}', index=False)
        print(f"File downloaded and saved at {file_path}.")
        return df
    else:
      df = pd.read_csv(file_path, dtype=dtype)
      print(f"File already exists at {file_path}, loading from storage.")
      return df
    
  else:
    print('URL not found.')

## Load and merge all layers, then add z-axis

In [ ]:
# define z-offset
offset = 1000000

# set scale factor
scale = 100000

# set start and end layer
start = 0
end = 31

# create emptt df to capture all layers
df_concat = pd.DataFrame()

for i in range(0,31):
  result = download_layer(i)
  
  if result is not None:
    
    # scale up
    result[['x', 'y']] = result[['x', 'y']].apply(lambda n: n * scale)
    
    # add z-axis to layer
    result['z'] = offset * i
    
    df_concat = pd.concat([df_concat, result], ignore_index=True)

df_concat

URL not found.
URL not found.
URL not found.
Found https://cdn.humanatlas.io/image-store/vccf-data-cell-nodes/unpublished/colon-xenium-stanford/layer_3-nodes.csv, downloading now.
File already exists at data/layer_3.csv, loading from storage.
Found https://cdn.humanatlas.io/image-store/vccf-data-cell-nodes/unpublished/colon-xenium-stanford/layer_4-nodes.csv, downloading now.
File already exists at data/layer_4.csv, loading from storage.
Found https://cdn.humanatlas.io/image-store/vccf-data-cell-nodes/unpublished/colon-xenium-stanford/layer_5-nodes.csv, downloading now.
File already exists at data/layer_5.csv, loading from storage.
Found https://cdn.humanatlas.io/image-store/vccf-data-cell-nodes/unpublished/colon-xenium-stanford/layer_6-nodes.csv, downloading now.
File already exists at data/layer_6.csv, loading from storage.
Found https://cdn.humanatlas.io/image-store/vccf-data-cell-nodes/unpublished/colon-xenium-stanford/layer_7-nodes.csv, downloading now.
File already exists at data/

,x,y,Cell Type,z
0,1.077960e+08,2.556820e+08,Immature Goblet,3000000
1,1.079010e+08,2.534230e+08,Tuft,3000000
2,1.082650e+08,2.554090e+08,TA1,3000000
3,1.091890e+08,2.547380e+08,Immature Goblet,3000000
4,1.253460e+08,2.554640e+08,CD4+,3000000
...,...,...,...,...
2639210,5.830022e+07,2.480927e+08,Immature Goblet,31000000
2639211,5.916007e+07,2.464072e+08,Immature Goblet,31000000
2639212,5.739870e+07,2.468947e+08,Immature Goblet,31000000
2639213,5.423405e+07,2.475479e+08,Goblet,31000000


## Visualize with `node-dist-vis`

In [86]:
# Next, let's define a function that turns a DataFrame into a node list that can then be passed into the CdeVisualization or NodeDistVis widget
def make_node_list(df: pd.DataFrame):
  """Turn a DataFrame into a list of dicts for passing them into a HRA widget

  Args:
      df (pd.DataFrame): A DataFrame with cells
  """

  node_list = [{'x': row['x'], 'y': row['y'], 'z': row['z'], 'Cell Type': row['Cell Type']}
               for index, row in df.iterrows()]

  return node_list

In [87]:
node_list = make_node_list(df_concat)
edge_list = [["Cell ID", "Target ID", "X1", "Y1", "Z1", "X2", "Y2", "Z2"]] # this makes an empty edges list

In [88]:
node_dist_vis = NodeDistVis(
    nodes=node_list,
    node_target_key="Cell Type",
    edges=edge_list
)
display(node_dist_vis)

NodeDistVis(color_map=None, color_map_data=None, color_map_key=None, color_map_keys=None, edge_keys=None, edge…